# Stereo Vision Assignment: 3D Reconstruction from Binocular Images


## Introduction

In this assignment, you will implement a complete binocular stereo vision pipeline — from projecting a 3D point cloud through two cameras, to finding corresponding points, computing disparity, and reconstructing 3D geometry.

### Learning Objectives

- Understand the perspective projection model and camera intrinsic/extrinsic parameters
- Construct a parallel-axis binocular stereo system
- Determine corresponding points between left and right images
- Compute disparity and generate a disparity map
- Perform 3D reconstruction using triangulation
- Evaluate reconstruction accuracy against ground truth

### Stereo Geometry Overview

```
        Left Camera        Right Camera
            O_L  ----B----  O_R
             |               |
             |   🐴 Horse   |
             |  Point Cloud  |
             v               v
         Left Image      Right Image
        (u_L, v_L)     (u_R, v_R)
        
        Disparity: d = u_L - u_R
        Depth:     Z = f·B / d
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# This magic command ensures plots appear inline in the notebook
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print("All libraries loaded successfully!")


## Step 1: Load and Visualize the Point Cloud

The file `horse.txt` contains a 3D point cloud of a horse model obtained from a 3D scanner. Each line has three floating-point numbers (x, y, z) separated by spaces, representing the coordinates of one point in meters.

- **Number of points:** 48,485
- **Bounding box:** approximately 8 cm × 18 cm × 15 cm

**Task:** Load the data and create a 3D scatter plot using `matplotlib`.


In [ ]:
# Load the horse point cloud
points = np.loadtxt('horse.txt')

print(f"Point cloud shape: {points.shape}")
print(f"Number of points: {len(points)}")
print(f"X range: [{points[:,0].min():.4f}, {points[:,0].max():.4f}] m")
print(f"Y range: [{points[:,1].min():.4f}, {points[:,1].max():.4f}] m")
print(f"Z range: [{points[:,2].min():.4f}, {points[:,2].max():.4f}] m")

# Create a 3D scatter plot of the point cloud
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points[:,0], points[:,1], points[:,2], c=points[:,2], cmap='viridis', s=0.1)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title('Horse Point Cloud (48485 points)')
plt.tight_layout()
plt.show()


## Step 2: The Pinhole Camera Model

A 3D point **P** in camera coordinates is projected to pixel coordinates $(u, v)$:

$$x_n = \frac{X_c}{Z_c}, \quad y_n = \frac{Y_c}{Z_c} \quad \text{(normalized image coordinates)}$$

$$u = \frac{f}{d_u} x_n + u_0, \quad v = \frac{f}{d_v} y_n + v_0 \quad \text{(pixel coordinates)}$$

The world-to-camera transform is:

$$\mathbf{P}_c = \mathbf{R} \mathbf{P}_w + \mathbf{T}$$

where $\mathbf{R}$ is the rotation matrix (Z-Y-X Euler angles) and $\mathbf{T}$ is the translation vector.

### Intrinsic Parameters

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Focal length | $f$ | Distance from lens to image plane |
| Principal point | $(u_0, v_0)$ | Optical center in pixel coordinates |
| Pixel size | $(d_u, d_v)$ | Physical size of one pixel |
| Distortion | $(k_1, k_2)$ | Radial distortion coefficients |

### Extrinsic Parameters

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Euler angles | $(\alpha, \beta, \theta)$ | Rotation around Z, Y, X axes |
| Translation | $\mathbf{T}$ | Camera position offset |

The `Camera` class below implements this model. Study the code carefully.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Camera:
    def __init__(self, f, u0, v0, du, dv, k1=0, k2=0):
        self.f = f
        self.u0 = u0
        self.v0 = v0
        self.du = du
        self.dv = dv
        self.k1 = k1
        self.k2 = k2
        
    def perspective_projection(self, points_3d, alpha, beta, theta, T):
        Rz = np.array([[np.cos(alpha), -np.sin(alpha), 0],
                       [np.sin(alpha), np.cos(alpha), 0],
                       [0, 0, 1]])
        Ry = np.array([[np.cos(beta), 0, np.sin(beta)],
                       [0, 1, 0],
                       [-np.sin(beta), 0, np.cos(beta)]])
        Rx = np.array([[1, 0, 0],
                       [0, np.cos(theta), -np.sin(theta)],
                       [0, np.sin(theta), np.cos(theta)]])
        R = Rz @ Ry @ Rx
        points_cam = (R @ points_3d.T).T + T
        valid_indices = points_cam[:, 2] > 0
        points_cam = points_cam[valid_indices]
        if len(points_cam) == 0:
            return np.array([]), valid_indices
        x_norm = points_cam[:, 0] / points_cam[:, 2]
        y_norm = points_cam[:, 1] / points_cam[:, 2]
        r2 = x_norm**2 + y_norm**2
        distortion = 1 + self.k1 * r2 + self.k2 * r2**2
        x_distorted = x_norm * distortion
        y_distorted = y_norm * distortion
        u = self.f / self.du * x_distorted + self.u0
        v = self.f / self.dv * y_distorted + self.v0
        points_2d = np.column_stack((u, v))
        return points_2d, valid_indices
    
    def _draw_circle(self, image, center_u, center_v, radius=3):
        height, width = image.shape
        center_u_int = int(round(center_u))
        center_v_int = int(round(center_v))
        u_min = max(0, center_u_int - radius)
        u_max = min(width, center_u_int + radius + 1)
        v_min = max(0, center_v_int - radius)
        v_max = min(height, center_v_int + radius + 1)
        for v in range(v_min, v_max):
            for u in range(u_min, u_max):
                distance = np.sqrt((u - center_u_int)**2 + (v - center_v_int)**2)
                if distance <= radius:
                    image[v, u] = 1
    
    def draw_image(self, points_2d, image_size=(1024, 768), point_radius=3):
        if len(points_2d) == 0:
            print("No valid projection points to display")
            return np.zeros((image_size[1], image_size[0]))
        image = np.zeros((image_size[1], image_size[0]))
        valid_points = []
        for point in points_2d:
            u, v = point
            if 0 <= u < image_size[0] and 0 <= v < image_size[1]:
                valid_points.append(point)
        valid_points = np.array(valid_points)
        if len(valid_points) == 0:
            print("All projection points are outside the image bounds")
            return image
        for point in valid_points:
            u, v = point
            self._draw_circle(image, u, v, point_radius)
        plt.figure(figsize=(6, 4))
        plt.imshow(image, cmap='gray', vmin=0, vmax=1)
        plt.title(f'Camera Projection Image\n({len(valid_points)} points with radius {point_radius} pixels)')
        plt.xlabel('u (pixels)')
        plt.ylabel('v (pixels)')
        plt.grid(False)
        plt.tight_layout()
        plt.show()
        print(f"Successfully drew {len(valid_points)} circular points on the image")
        print(f"Image size: {image_size[0]} x {image_size[1]} pixels")
        print(f"Point radius: {point_radius} pixels")
        return image


## Step 3: Construct a Binocular Stereo System

In a **parallel-axis stereo system**, two identical cameras are separated by a baseline $B$ along the X-axis. Both cameras look in the same direction ($-Z_{world}$).

- Left camera at $C_L = (-B/2,\ 0,\ D)$
- Right camera at $C_R = (B/2,\ 0,\ D)$

The key relationship between disparity and depth is:

$$d = u_L - u_R = \frac{f_{pixel} \cdot B}{Z}$$

```
World Coordinate System:
        Y (up)
        |
        |    / Z (toward cameras)
        |   /
        |  /
        | /
        +--------- X (right)
        
Camera positions (top view):
        
   Z_world ←←←←←←←←←←←←←← (cameras look this way)
              |         |
         C_L  |  Horse  |  C_R
         (-B/2)         (+B/2)
              |         |
              ←——— B ———→
```

**Task:** Set up camera parameters, project the point cloud through both cameras, and display the resulting images.

> **Note on parameter choice:** We use $f = 8$ mm instead of a more typical $f = 50$ mm because the horse model is only ~8 cm across. A standard lens at close range would produce a very narrow field of view, cropping most of the point cloud. The shorter focal length gives a wider field of view appropriate for this small-scale object at a distance of $D = 0.35$ m.


In [ ]:
# ============================================================
# Camera Intrinsic Parameters (adjusted for the small horse model)
# ============================================================
f = 8               # focal length (mm)
u0, v0 = 512, 384   # principal point (pixels) — image center for 1024×768
du = dv = 0.01      # pixel size (mm/pixel)
k1, k2 = 0, 0       # no distortion (simplified for this exercise)

# Create two cameras with IDENTICAL intrinsic parameters
cam_left = Camera(f=f, u0=u0, v0=v0, du=du, dv=dv, k1=k1, k2=k2)
cam_right = Camera(f=f, u0=u0, v0=v0, du=du, dv=dv, k1=k1, k2=k2)

# ============================================================
# Stereo Geometry Parameters
# ============================================================
B = 0.06    # baseline (meters) — distance between the two cameras
D = 0.35    # distance from cameras to point cloud center (meters)

# ============================================================
# Camera Extrinsic Parameters
# ============================================================
# Both cameras look along -Z_world direction
# This requires a 180° rotation around the X-axis (theta = π)
alpha = 0          # rotation around Z (radians)
beta = 0           # rotation around Y (radians)
theta = np.pi      # rotation around X (radians) — 180° to face the point cloud

# Translation vectors: T = -R @ C
# where C is the camera center in world coordinates
# Left camera center in world:  C_L = (-B/2, 0, D)
# Right camera center in world: C_R = (B/2, 0, D)
#
# TODO: Compute the translation vectors
# Hint: Build the rotation matrix R = Rz(alpha) @ Ry(beta) @ Rx(theta),
# then compute T = -R @ C for each camera.

# Build rotation matrix
Rz_s = ??????????????????????????????
Ry_s = ??????????????????????????????
Rx_s = ??????????????????????????????
R_s = ???????????????????????????????

# Camera centers in world coordinates
C_left  = np.array([-B/2, 0, D])   # left camera: shifted to the left
C_right = np.array([B/2, 0, D])    # right camera: shifted to the right

# Translation vectors: T = -R @ C
T_left  = ???????????????????
T_right = ???????????????????

# Verify your translations
print(f"T_left  = {T_left}")
print(f"T_right = {T_right}")
print(f"Expected: T_left ≈ [0.03, 0, 0.35], T_right ≈ [-0.03, 0, 0.35]")

# ============================================================
# Project the point cloud through both cameras
# ============================================================
pts2d_left, valid_left = cam_left.perspective_projection(
    points, alpha, beta, theta, T_left)
pts2d_right, valid_right = cam_right.perspective_projection(
    points, alpha, beta, theta, T_right)

print(f"\nLeft camera:  {len(pts2d_left)} visible points / {len(points)} total")
print(f"Right camera: {len(pts2d_right)} visible points / {len(points)} total")

# ============================================================
# Display left and right images side by side
# ============================================================
def generate_projection_image(points_2d, image_size=(1024, 768), point_radius=2):
    """Generate a binary image array from 2D projected points (without displaying)."""
    image = np.zeros((image_size[1], image_size[0]))
    for point in points_2d:
        u, v = point
        if 0 <= u < image_size[0] and 0 <= v < image_size[1]:
            u_min = max(0, int(round(u)) - point_radius)
            u_max = min(image_size[0], int(round(u)) + point_radius + 1)
            v_min = max(0, int(round(v)) - point_radius)
            v_max = min(image_size[1], int(round(v)) + point_radius + 1)
            for vi in range(v_min, v_max):
                for ui in range(u_min, u_max):
                    if np.sqrt((ui - round(u))**2 + (vi - round(v))**2) <= point_radius:
                        image[vi, ui] = 1
    return image

img_left = generate_projection_image(pts2d_left, point_radius=2)
img_right = generate_projection_image(pts2d_right, point_radius=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(img_left, cmap='gray')
ax1.set_title('Left Image')
ax1.set_xlabel('u (pixels)')
ax1.set_ylabel('v (pixels)')
ax2.imshow(img_right, cmap='gray')
ax2.set_title('Right Image')
ax2.set_xlabel('u (pixels)')
ax2.set_ylabel('v (pixels)')
plt.suptitle('Binocular Stereo Image Pair', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n💡 Observe: The horse in the left image appears slightly shifted to the RIGHT")
print(f"   compared to the right image. This horizontal shift is the DISPARITY.")


## Step 4: Finding Corresponding Points

In a real stereo system, finding corresponding points between left and right images is a challenging problem (stereo matching). However, in this assignment, we have a simplifying advantage: we know the 3D-to-2D mapping for each camera.

**Key insight:** `valid_indices` tells us which 3D points project into each camera. Two points in the left and right images are *corresponding* if they come from the **same 3D point**.

### Algorithm Outline

1. Get the original point indices visible in the left camera: `left_visible_idx = np.where(valid_left)[0]`
2. Get the original point indices visible in the right camera: `right_visible_idx = np.where(valid_right)[0]`
3. Find common indices (visible in both): `common_idx = np.intersect1d(left_visible_idx, right_visible_idx)`
4. Map common indices back to positions in `pts2d_left` and `pts2d_right` using `np.searchsorted`

**Task:** Implement this correspondence algorithm.


In [ ]:
# ============================================================
# Step 4: Find corresponding points between left and right images
# ============================================================

# Get the original point indices that are visible in each camera
# valid_left[i] is True if point i is in front of the left camera
left_visible_idx = np.where(valid_left)[0]    # sorted array of indices
right_visible_idx = np.where(valid_right)[0]   # sorted array of indices

print(f"Points visible in left camera:  {len(left_visible_idx)}")
print(f"Points visible in right camera: {len(right_visible_idx)}")

# Find the indices of 3D points that are visible in BOTH cameras
common_idx = np.intersect1d(left_visible_idx, right_visible_idx)

print(f"Points visible in BOTH cameras: {len(common_idx)}")

# TODO: Map each common index to its position in the left/right projected point arrays
# Since left_visible_idx and right_visible_idx are sorted, use np.searchsorted
# For example: if common_idx[k] = 42, and left_visible_idx = [5, 12, 42, 67, ...],
# then the position of 42 in left_visible_idx is 2, and pts2d_left[2] is its projection.

# Find positions of common points in the left projected array
left_pos = ?????????????????????????????

# Find positions of common points in the right projected array
right_pos = ????????????????????????????

# Extract the corresponding point pairs
left_matches = pts2d_left[left_pos]      # (M, 2) — pixel coords in left image
right_matches = pts2d_right[right_pos]   # (M, 2) — pixel coords in right image
gt_3d = points[common_idx]               # (M, 3) — ground truth 3D coordinates

print(f"\nNumber of corresponding point pairs: {len(left_matches)}")
print(f"Left match example (first 3):  \n{left_matches[:3]}")
print(f"Right match example (first 3): \n{right_matches[:3]}")

# ============================================================
# Visualize: Draw corresponding points on both images
# ============================================================
# Select a subset of points to visualize (every 500th point)
vis_step = 500
vis_indices = np.arange(0, len(left_matches), vis_step)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(img_left, cmap='gray')
ax2.imshow(img_right, cmap='gray')

colors = plt.cm.rainbow(np.linspace(0, 1, len(vis_indices)))
for i, idx in enumerate(vis_indices):
    u_l, v_l = left_matches[idx]
    u_r, v_r = right_matches[idx]
    ax1.plot(u_l, v_l, 'o', color=colors[i], markersize=6)
    ax1.annotate(str(i), (u_l, v_l), color=colors[i], fontsize=8, fontweight='bold')
    ax2.plot(u_r, v_r, 'o', color=colors[i], markersize=6)
    ax2.annotate(str(i), (u_r, v_r), color=colors[i], fontsize=8, fontweight='bold')

ax1.set_title('Left Image with Corresponding Points')
ax1.set_xlabel('u (pixels)'); ax1.set_ylabel('v (pixels)')
ax2.set_title('Right Image with Corresponding Points')
ax2.set_xlabel('u (pixels)'); ax2.set_ylabel('v (pixels)')
plt.suptitle('Corresponding Points (same color/number = same 3D point)', fontsize=13)
plt.tight_layout()
plt.show()

print("💡 Notice: Corresponding points have similar v-coordinates but different u-coordinates.")
print("   The horizontal difference (u_L - u_R) is the disparity.")


## Step 5: Disparity Computation

For a parallel-axis stereo system, corresponding points lie on the same horizontal scanline (same $v$-coordinate). The **disparity** is defined as:

$$d = u_L - u_R$$

The disparity is inversely proportional to depth:

$$d = \frac{f_{pixel} \cdot B}{Z}$$

- **Larger disparity** → closer object
- **Smaller disparity** → farther object

**Task:** Compute disparity for all corresponding points and visualize.


In [ ]:
# ============================================================
# Step 5: Compute disparity for all corresponding points
# ============================================================

# Compute the disparity for each corresponding point pair
# Disparity = u_left - u_right
disparity = ??????????????????????????

print(f"Disparity statistics:")
print(f"  Min:  {disparity.min():.2f} pixels")
print(f"  Max:  {disparity.max():.2f} pixels")
print(f"  Mean: {disparity.mean():.2f} pixels")
print(f"  Std:  {disparity.std():.2f} pixels")

# ============================================================
# Generate and display a disparity map
# ============================================================
# Create a 2D disparity map aligned with the left image
disparity_map = np.full((768, 1024), np.nan)  # initialize with NaN
u_coords = np.round(left_matches[:, 0]).astype(int)
v_coords = np.round(left_matches[:, 1]).astype(int)

# Only keep points within image bounds
mask = (u_coords >= 0) & (u_coords < 1024) & (v_coords >= 0) & (v_coords < 768)
for i in np.where(mask)[0]:
    disparity_map[v_coords[i], u_coords[i]] = disparity[i]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Disparity map
im = ax1.imshow(disparity_map, cmap='jet', vmin=np.nanmin(disparity_map), 
                vmax=np.nanmax(disparity_map))
ax1.set_title('Disparity Map')
ax1.set_xlabel('u (pixels)'); ax1.set_ylabel('v (pixels)')
plt.colorbar(im, ax=ax1, label='Disparity (pixels)')

# Disparity histogram
valid_disp = disparity[disparity > 0]
ax2.hist(valid_disp, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
ax2.set_xlabel('Disparity (pixels)')
ax2.set_ylabel('Number of points')
ax2.set_title('Disparity Distribution')
ax2.axvline(np.mean(valid_disp), color='red', linestyle='--', label=f'Mean = {np.mean(valid_disp):.1f}')
ax2.legend()

plt.tight_layout()
plt.show()

# Verify against theoretical value
f_pixel = f / du
Z_center = D  # approximate depth of horse center
d_theoretical = f_pixel * B / Z_center
print(f"\n📊 Theoretical disparity at Z={D}m: d = f·B/Z = {f_pixel}×{B}/{Z_center} = {d_theoretical:.1f} pixels")
print(f"   Measured mean disparity: {np.mean(valid_disp):.1f} pixels")


## Step 6: 3D Reconstruction and Error Analysis

Using the disparity and camera parameters, we can reconstruct the 3D coordinates of each point using **triangulation**:

$$Z = \frac{f_{pixel} \cdot B}{d}$$

$$X = \frac{(u_L - u_0) \cdot Z}{f_{pixel}}$$

$$Y = \frac{(v_L - v_0) \cdot Z}{f_{pixel}}$$

These coordinates are in the **left camera frame**. To compare with the original point cloud (world frame), we need to transform back:

$$\mathbf{P}_{world} = \mathbf{R}^{-1} (\mathbf{P}_{cam} - \mathbf{T}_{left})$$

Since $\mathbf{R}$ is a rotation matrix, $\mathbf{R}^{-1} = \mathbf{R}^T$.

**Task:** Implement the reconstruction and compute errors.


In [ ]:
# ============================================================
# Step 6: 3D Reconstruction from disparity
# ============================================================

# Compute f_pixel (focal length in pixels)
f_pixel = f / du  # = 800 pixels

# TODO: Reconstruct 3D coordinates in the LEFT CAMERA FRAME
# For each corresponding point pair:
#   d = u_L - u_R  (disparity)
#   Z = f_pixel * B / d
#   X = (u_L - u0) * Z / f_pixel
#   Y = (v_L - v0) * Z / f_pixel

# Reconstruct depth: Z = f_pixel * B / d
Z_recon = ??????????????????????????????

# Reconstruct X: X = (u_L - u0) * Z / f_pixel
X_recon = ????????????????????????????

# Reconstruct Y: Y = (v_L - v0) * Z / f_pixel
Y_recon = ?????????????????????????????

# Assemble reconstructed points in camera frame
reconstructed_cam = np.column_stack([X_recon, Y_recon, Z_recon])
print(f"Reconstructed {len(reconstructed_cam)} points in camera frame")
print(f"Z range: [{Z_recon.min():.4f}, {Z_recon.max():.4f}] m")

# ============================================================
# Transform from camera frame back to world frame
# ============================================================
# Build the rotation matrix (same as used in Camera class)
Rz_mat = np.array([[np.cos(alpha), -np.sin(alpha), 0],
                    [np.sin(alpha),  np.cos(alpha), 0],
                    [0, 0, 1]])
Ry_mat = np.array([[np.cos(beta), 0, np.sin(beta)],
                    [0, 1, 0],
                    [-np.sin(beta), 0, np.cos(beta)]])
Rx_mat = np.array([[1, 0, 0],
                    [0, np.cos(theta), -np.sin(theta)],
                    [0, np.sin(theta),  np.cos(theta)]])
R = Rz_mat @ Ry_mat @ Rx_mat

# Inverse rotation: R^(-1) = R^T for rotation matrices
R_inv = R.T

# Transform reconstructed points from camera frame to world frame
# P_world = R_inv @ (P_cam - T_left)
reconstructed_world = (R_inv @ (reconstructed_cam - T_left).T).T

print(f"\nReconstructed world coordinates (first 3 points):")
print(reconstructed_world[:3])
print(f"\nGround truth (first 3 points):")
print(gt_3d[:3])

# ============================================================
# Compute reconstruction errors
# ============================================================
# Compute per-point Euclidean distance between reconstructed and ground truth
errors = np.linalg.norm(reconstructed_world - gt_3d, axis=1)

# Statistics
mean_error = np.mean(errors)
median_error = np.median(errors)
max_error = np.max(errors)
rmse = np.sqrt(np.mean(errors**2))

print(f"\n{'='*50}")
print(f"  RECONSTRUCTION ERROR ANALYSIS")
print(f"{'='*50}")
print(f"  Mean Error:   {mean_error*1000:.4f} mm")
print(f"  Median Error: {median_error*1000:.4f} mm")
print(f"  RMSE:         {rmse*1000:.4f} mm")
print(f"  Max Error:    {max_error*1000:.4f} mm")
print(f"  Min Error:    {np.min(errors)*1000:.6f} mm")
print(f"{'='*50}")

# ============================================================
# Visualization
# ============================================================
fig = plt.figure(figsize=(18, 10))

# 1. Error distribution
ax1 = fig.add_subplot(231)
ax1.hist(errors*1000, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
ax1.set_xlabel('Error (mm)')
ax1.set_ylabel('Count')
ax1.set_title('Error Distribution')
ax1.axvline(rmse*1000, color='red', linestyle='--', label=f'RMSE={rmse*1000:.4f}mm')
ax1.legend()

# 2. Error vs. depth
ax2 = fig.add_subplot(232)
# Depth of each point in left camera frame
Z_cam = (R @ gt_3d.T).T[:, 2] + T_left[2]
ax2.scatter(Z_cam*100, errors*1000, s=0.1, alpha=0.3, c='steelblue')
ax2.set_xlabel('Depth Z (cm)')
ax2.set_ylabel('Error (mm)')
ax2.set_title('Error vs. Depth')

# 3. Disparity vs. depth (verify inverse relationship)
ax3 = fig.add_subplot(233)
ax3.scatter(Z_cam*100, disparity, s=0.1, alpha=0.3, c='green')
Z_range = np.linspace(Z_cam.min(), Z_cam.max(), 100)
ax3.plot(Z_range*100, f_pixel*B/Z_range, 'r-', linewidth=2, label=f'd = fB/Z')
ax3.set_xlabel('Depth Z (cm)')
ax3.set_ylabel('Disparity (pixels)')
ax3.set_title('Disparity vs. Depth')
ax3.legend()

# 4. Original vs reconstructed (3D)
ax4 = fig.add_subplot(234, projection='3d')
ax4.scatter(gt_3d[:,0]*100, gt_3d[:,1]*100, gt_3d[:,2]*100, 
            s=0.1, alpha=0.3, c='blue', label='Ground Truth')
ax4.scatter(reconstructed_world[:,0]*100, reconstructed_world[:,1]*100, 
            reconstructed_world[:,2]*100, s=0.1, alpha=0.3, c='red', label='Reconstructed')
ax4.set_xlabel('X (cm)'); ax4.set_ylabel('Y (cm)'); ax4.set_zlabel('Z (cm)')
ax4.set_title('Original vs Reconstructed')
ax4.legend()

# 5. Reconstructed colored by error
ax5 = fig.add_subplot(235, projection='3d')
sc = ax5.scatter(reconstructed_world[:,0]*100, reconstructed_world[:,1]*100, 
                 reconstructed_world[:,2]*100, s=0.5, c=errors*1000, cmap='hot', 
                 vmin=0, vmax=np.percentile(errors*1000, 95))
ax5.set_xlabel('X (cm)'); ax5.set_ylabel('Y (cm)'); ax5.set_zlabel('Z (cm)')
ax5.set_title('Reconstructed (colored by error)')
plt.colorbar(sc, ax=ax5, label='Error (mm)')

# 6. Point-wise error histogram on log scale
ax6 = fig.add_subplot(236)
ax6.hist(errors*1000, bins=50, color='steelblue', edgecolor='black', alpha=0.8, log=True)
ax6.set_xlabel('Error (mm)')
ax6.set_ylabel('Count (log scale)')
ax6.set_title('Error Distribution (log scale)')

plt.tight_layout()
plt.show()


## Bonus: Effect of Measurement Noise on Reconstruction

In a real stereo system, pixel coordinates are quantized and noisy. Let's simulate this by adding Gaussian noise to the pixel coordinates and observe how it affects reconstruction accuracy.

The depth error scales as:

$$\delta Z \approx \frac{Z^2}{f_{pixel} \cdot B} \cdot \delta d$$


In [ ]:
# ============================================================
# Bonus: Add pixel noise and analyze its effect
# ============================================================
noise_levels = [0.0, 0.5, 1.0, 2.0]  # standard deviation in pixels

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, sigma in enumerate(noise_levels):
    # Add Gaussian noise to pixel coordinates
    np.random.seed(42)
    left_noisy = left_matches + np.random.randn(*left_matches.shape) * sigma
    right_noisy = right_matches + np.random.randn(*right_matches.shape) * sigma
    
    # Compute noisy disparity
    d_noisy = left_noisy[:, 0] - right_noisy[:, 0]
    d_noisy = np.maximum(d_noisy, 0.1)  # avoid division by zero
    
    # Reconstruct with noisy data
    Z_n = ????????????????????????
    X_n = ?????????????????????????????
    Y_n = ????????????????????????????
    
    recon_cam_n = np.column_stack([X_n, Y_n, Z_n])
    recon_world_n = (R_inv @ (recon_cam_n - T_left).T).T
    
    # Compute errors
    errors_n = np.linalg.norm(recon_world_n - gt_3d, axis=1)
    rmse_n = np.sqrt(np.mean(errors_n**2))
    
    axes[i].hist(errors_n*1000, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].set_xlabel('Error (mm)')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'Noise σ = {sigma} pixels → RMSE = {rmse_n*1000:.3f} mm')
    axes[i].axvline(rmse_n*1000, color='red', linestyle='--')

plt.suptitle('Effect of Pixel Noise on Reconstruction Accuracy', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 Key insight: Reconstruction error grows rapidly with pixel noise.")
print("   This is because depth error δZ ≈ Z²/(f·B) × δd — it scales quadratically with depth!")


## Discussion Questions

1. **Baseline and Disparity:** If you increase the baseline $B$, what happens to the disparity? What happens to the reconstruction accuracy? What are the practical limitations of a very large baseline?

2. **Depth Resolution:** Why does the reconstruction error increase for points that are farther from the camera? Derive the relationship between depth error $\delta Z$ and depth $Z$.

3. **Distortion Effect:** If you set $k_1 = -0.1$ in the Camera class (radial distortion), how would the reconstruction results change? Would our simple triangulation formula still be valid?

4. **Real-World Correspondence:** In this assignment, we used ground truth to find corresponding points. In a real stereo system, how would you find correspondences? What challenges arise?

5. **Parameter Design:** If the horse model were 10× larger (a real horse ~2 m tall), how would you adjust the camera parameters? Justify your choices.
